# tpu-hawkeye on Kaggle TPU
Settings: Accelerator = TPU v5e-8, Internet = On, Secrets: add `LLM_API_KEY`. Run cells top to bottom.

In [ ]:
%cd /kaggle/working
!rm -rf tpu-hawkeye
!git clone --recurse-submodules https://github.com/m8ngotree/tpu-hawkeye.git
%cd tpu-hawkeye
!pip install -q openai

## Step 1 - verify the TPU and the taxonomy cells

In [ ]:
import jax
print(jax.devices())   # expect 8 TPU devices; if not, stop and fix this first

In [ ]:
!python scripts/verify_cells.py

## Step 2 - smoke-test the agent (one workload, few turns, one condition)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["LLM_API_KEY"] = UserSecretsClient().get_secret("LLM_API_KEY")

In [ ]:
!python -m eval.run_agent_eval --workloads 12p_RMSNorm --conditions taxonomy --max-turns 10 --tag smoke

## Step 3 - pilot (edit workloads/turns as needed)

In [ ]:
!python -m eval.run_agent_eval --workloads 12p_RMSNorm,8p_GEMM,41k_Gemm_Add_ReLU --conditions taxonomy none --reps 1 --max-turns 30 --tag pilot

## Save results (Kaggle wipes the session when it ends)

In [ ]:
!cd /kaggle/working && zip -rq tpu-hawkeye-results.zip tpu-hawkeye/results && ls -la tpu-hawkeye-results.zip